# 🚀 LinkedIn CEO / Executive Finder

> Công cụ tự động tìm kiếm, xác thực danh tính **CEO, Founder, Chủ tịch** trên LinkedIn từ **Tên miền website** hoặc **Từ khóa ngành nghề**, tự động lưu và làm đẹp **Google Sheet 8 cột** (tích hợp Xác Minh AI).


In [ ]:
#@title 📦 Cài đặt thư viện { display-mode: "form" }
#@markdown *Bắt đầu tải và cài đặt các thư viện cần thiết cho Tool.*

import sys
!pip install -q curl-cffi selectolax gspread google-auth requests orjson
print("✅ Cài đặt thư viện thành công!")


In [ ]:
#@title 🔑 Kết nối tài khoản Google { display-mode: "form" }
#@markdown *Bạn sẽ cần đăng nhập và cấp quyền cho Colab, nhớ để cho Sheet chia sẽ ở chế độ **"Người chỉnh sửa"** nhé*

import sys, os
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    try:
        from google.colab import auth
        print("⏳ Đang mở cửa sổ xác thực Google...")
        auth.authenticate_user()
        print("\n🎉 ĐĂNG NHẬP THÀNH CÔNG! Google Colab đã có quyền đọc.")
    except Exception as e:
        print(f"❌ Lỗi khi đăng nhập Google Colab: {e}")
else:
    print("ℹ️ Bạn đang chạy ở máy cá nhân (Local). Tự động dùng Service Account hoặc Token nếu có.")


In [ ]:
#@title 🚀 BỘ XỬ LÝ TRUNG TÂM (CORE ENGINE) { display-mode: "form" }
#@markdown Hỗ trợ:
#@markdown - Tìm kiếm đa tầng (Multi-tier Resolution) danh tính CEO qua Domain Website.
#@markdown - Tìm kiếm hàng loạt CEO / Founder theo Từ khóa ngành nghề (Keyword Mode).
#@markdown - Xác minh tính chính xác bằng Google Gemini / Gemma AI (AI Verification).
#@markdown - Tự động kết nối, định dạng, lưu trữ và chống trùng lặp trên Google Sheets 8 cột.


import os
import sys
import re
import time
import urllib.parse
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Set, Dict, Any, Union, Tuple
from dataclasses import dataclass, field

import orjson
import requests
import gspread
from curl_cffi import requests as curl_requests
from selectolax.parser import HTMLParser

# Nhận diện môi trường Google Colab
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


# ==============================================================================
# 1. ⚙️ CẤU HÌNH & HẰNG SỐ HỆ THỐNG
# ==============================================================================

# Các đường dẫn trang ban lãnh đạo phổ biến trên website doanh nghiệp
LEADERSHIP_PATHS = [
    "/leadership", "/team", "/about-us", "/about", "/our-team",
    "/board-of-directors", "/ban-lanh-dao", "/gioi-thieu", "/people", "/management"
]

# Từ khóa nhận diện chức danh lãnh đạo cấp cao
CEO_KEYWORDS = [
    "ceo", "chief executive", "founder", "co-founder",
    "tổng giám đốc", "giám đốc điều hành", "chủ tịch"
]

# Tiền tố chỉ chức danh cũ cần loại bỏ
BAD_ROLE_PREFIXES = [
    "ex-", "ex ", "former", "cựu", "previously", "past ",
    "advisor", "investor", "intern", "assistant"
]

# Danh mục chức danh hợp lệ
VALID_LEADERSHIP_ROLES = [
    "ceo", "chief executive", "founder", "co-founder", "cofounder",
    "tổng giám đốc", "giám đốc điều hành", "chủ tịch", "president", "managing director"
]

# Chuẩn hóa tên thương hiệu lớn để tăng độ chính xác khi tìm kiếm
BRAND_MAP = {
    "Google": "Google",
    "Microsoft": "Microsoft",
    "Nvidia": "NVIDIA",
    "Amd": "AMD",
    "Amazon": "Amazon",
    "Jetbrains": "JetBrains",
    "Apple": "Apple",
    "Meta": "Meta",
    "Openai": "OpenAI",
    "Vng": "VNG",
    "Fpt": "FPT"
}

# Cấu trúc tiêu đề và độ rộng cột trên Google Sheet chuẩn
SHEET_HEADERS = [
    "STT", "CEO Name", "Role / Title", "Keyword",
    "Company", "LinkedIn URL", "Xác Minh AI", "Ngày Cập Nhật"
]
SHEET_COL_WIDTHS = [55, 200, 220, 120, 200, 320, 320, 160]


def get_secret(key_names: Union[str, List[str]], default: str = "") -> str:
    """Tự động tìm kiếm Secret Key từ Google Colab Secrets (userdata) hoặc Biến môi trường (os.environ)."""
    if isinstance(key_names, str):
        key_names = [key_names]

    # Thử lấy từ Colab userdata (biểu tượng chìa khóa 🔑)
    try:
        from google.colab import userdata
        for key in key_names:
            try:
                val = userdata.get(key)
                if val is not None and str(val).strip():
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass

    # Thử lấy từ biến môi trường hệ thống
    for key in key_names:
        val = os.environ.get(key)
        if val is not None and str(val).strip():
            return str(val).strip()

    return default


def now_hcm_str() -> str:
    """Trả về thời gian hiện tại định dạng YYYY-MM-DD HH:MM:SS theo múi giờ TP. Hồ Chí Minh."""
    try:
        import zoneinfo
        return datetime.now(zoneinfo.ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# ==============================================================================
# 2. 📦 MÔ HÌNH DỮ LIỆU (DATA MODEL)
# ==============================================================================

@dataclass
class CEOLead:
    """Đại diện cho thông tin một khách hàng tiềm năng / Lãnh đạo cấp cao tìm thấy."""
    name: str
    role: str
    keyword: str
    company: str = "N/A"
    linkedin_link: str = ""
    ai_status: str = "Chưa xác minh"
    updated_at: str = field(default_factory=now_hcm_str)

    @property
    def url(self) -> str:
        return self.linkedin_link

    def __getitem__(self, item: str) -> Any:
        return self.linkedin_link if item == "url" else getattr(self, item)

    def get(self, item: str, default: Any = None) -> Any:
        try:
            return self[item]
        except (AttributeError, KeyError):
            return default

    def to_dict(self) -> Dict[str, Any]:
        return {
            "name": self.name.strip(),
            "role": self.role.strip(),
            "keyword": self.keyword.strip(),
            "company": self.company.strip() or "N/A",
            "linkedin_link": self.linkedin_link.strip(),
            "ai_status": self.ai_status.strip() or "Chưa xác minh",
            "updated_at": self.updated_at,
            "url": self.linkedin_link.strip(),
        }


# ==============================================================================
# 3. 🌐 MẠNG & HTTP REQUEST (NETWORK UTILITIES)
# ==============================================================================

def _mask_proxy(proxy_url: Optional[str]) -> str:
    """Ẩn mật khẩu trong chuỗi proxy để log an toàn."""
    if not proxy_url:
        return "None"
    return re.sub(r"://([^:]+):([^@]+)@", r"://\1:***@", proxy_url)


def http_get(
    url: str,
    proxy: Optional[str] = None,
    timeout: int = 15,
    retries: int = 2,
    delay_between_retries: float = 0.5
) -> Optional[HTMLParser]:
    """Gửi HTTP GET Request giả lập Chrome TLS Fingerprint với cơ chế Fallback và Retry tự động."""
    effective_proxy = None if IN_COLAB and not proxy else proxy

    for attempt in range(1, retries + 1):
        # 1. Ưu tiên curl_cffi giả lập Chrome 120 (vượt qua hầu hết bot protection)
        try:
            response = curl_requests.get(
                url,
                proxy=effective_proxy,
                impersonate="chrome120",
                timeout=timeout
            )
            if response.status_code == 200 and response.text:
                if "We had temporary problems searching for web pages" not in response.text:
                    return HTMLParser(response.text)
            time.sleep(delay_between_retries)
        except Exception:
            # 2. Fallback sang thư viện requests tiêu chuẩn
            try:
                headers = {
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/122.0.0.0 Safari/537.36"
                }
                proxies_dict = {"http": effective_proxy, "https": effective_proxy} if effective_proxy else None
                fallback_resp = requests.get(
                    url,
                    headers=headers,
                    proxies=proxies_dict,
                    timeout=timeout
                )
                if fallback_resp.status_code == 200 and fallback_resp.text:
                    return HTMLParser(fallback_resp.text)
            except Exception:
                pass
            time.sleep(delay_between_retries)

    # 3. Fallback cuối cùng: Thử kết nối trực tiếp không qua proxy nếu proxy gặp lỗi
    if effective_proxy is not None:
        try:
            direct_resp = curl_requests.get(url, proxy=None, impersonate="chrome120", timeout=timeout)
            if direct_resp.status_code == 200 and direct_resp.text:
                return HTMLParser(direct_resp.text)
        except Exception:
            pass

    return None


def unquote_yahoo_url(raw_href: str) -> str:
    """Giải mã URL chuyển hướng của Yahoo Search để trích xuất link LinkedIn đích."""
    match = re.search(r"RU=([^/&?]+)", raw_href, re.IGNORECASE)
    if match:
        decoded = urllib.parse.unquote(match.group(1))
    else:
        decoded = raw_href
    return decoded.split("?")[0].rstrip("/")


# ==============================================================================
# 4. 🔍 PHÂN TÍCH CHỨC DANH & LINKEDIN (TITLE & ROLE PARSER)
# ==============================================================================

def is_leadership_role(role_text: Optional[str]) -> bool:
    """Kiểm tra xem chuỗi chức danh có thuộc vai trò lãnh đạo cấp cao hợp lệ hay không."""
    if not role_text:
        return False
    role_lower = role_text.lower()

    # Loại bỏ nếu chứa từ khóa cựu/nghỉ việc
    if any(bad in role_lower for bad in BAD_ROLE_PREFIXES):
        return False

    return any(valid in role_lower for valid in VALID_LEADERSHIP_ROLES)


def parse_linkedin_title(title: str, snippet: str = "") -> Tuple[str, str, str]:
    """Phân tách tiêu đề trang cá nhân LinkedIn thành (Tên, Chức Danh, Tên Công Ty)."""
    # Xóa hậu tố LinkedIn
    clean_title = re.sub(r"\s*[\-|\-|\/|\|]\s*LinkedIn.*$", "", title, flags=re.IGNORECASE).strip()
    parts = [p.strip() for p in re.split(r"\s+[-–|]\s+", clean_title) if p.strip()]

    if not parts:
        return "", "", "N/A"

    name = parts[0]
    # Kiểm tra tính hợp lệ của tên (tránh nhầm slogan hoặc danh xưng dài)
    if len(name.split()) > 7 or any(k in name.lower() for k in ["ceo", "founder", "hiring"]):
        name = ""

    role = ""
    company = ""

    # Duyệt các phần còn lại để tách chức danh và công ty
    for part in parts[1:]:
        if any(bad in part.lower() for bad in BAD_ROLE_PREFIXES):
            continue

        split_role = re.split(r"\s*(?:@|\bat\b|\bof\b)\s*", part, maxsplit=1, flags=re.IGNORECASE)
        candidate_role = split_role[0].strip()

        if is_leadership_role(candidate_role):
            role = candidate_role
            if len(split_role) == 2:
                company = split_role[1].strip()
            break

    # Nếu chưa tìm thấy chức danh qua @ / at / of, tìm theo cụm role trực tiếp
    if not role:
        for part in parts[1:]:
            if is_leadership_role(part):
                role = part
                break

    if not is_leadership_role(role):
        return name, "", "N/A"

    # Nếu công ty chưa có, trích xuất thêm từ snippet tìm kiếm
    if not company and snippet:
        match_exp = re.search(
            r"(?:Experience:\s*([^·\n]+)|(?:at|of)\s+([A-Z0-9][a-zA-Z0-9\s\.\-]{2,30}))",
            snippet
        )
        if match_exp:
            company = (match_exp.group(1) or match_exp.group(2) or "").strip()

    if company:
        company = re.sub(r"\.\.\.$", "", re.split(r"\s*[•|–—]\s*", company)[0]).strip()

    return name, role, company or "N/A"


def is_duplicate_ceo(clean_url: str, name: str, seen_set: Optional[Set[str]]) -> bool:
    """Kiểm tra xem URL hoặc Tên CEO đã tồn tại trong tập dữ liệu hay chưa."""
    if not seen_set:
        return False

    norm_url = (clean_url or "").strip().split("?")[0].rstrip("/").lower()
    if norm_url and norm_url in seen_set:
        return True

    match_username = re.search(r"linkedin\.com/in/([^/]+)", norm_url, re.IGNORECASE)
    if match_username and match_username.group(1).lower() in seen_set:
        return True

    return bool(name and name.strip().lower() in seen_set)


def print_lead(lead: Union[CEOLead, Dict[str, Any]], index: int, prefix: str = "") -> None:
    """In thông tin Lead ra màn hình console một cách trực quan, chuyên nghiệp."""
    lead_dict = lead.to_dict() if isinstance(lead, CEOLead) else lead
    tag = f"[{prefix}] " if prefix else ""
    link = lead_dict.get("linkedin_link") or lead_dict.get("url") or "N/A"
    ai_info = f" | {lead_dict.get('ai_status')}" if lead_dict.get("ai_status") else ""
    print(
        f" + [#{index}] {tag}{lead_dict.get('name')} | "
        f"{lead_dict.get('role')} | {lead_dict.get('company')} -> {link}{ai_info}"
    )


# ==============================================================================
# 5. 🧠 XÁC MINH DANH TÍNH VỚI GEMINI / GEMMA AI
# ==============================================================================

def extract_json_from_ai_response(text: str) -> Optional[dict]:
    """Bóc tách đối tượng JSON một cách an toàn từ kết quả text của mô hình AI."""
    if not text:
        return None

    # 1. Tìm kiếm trong khối code block ```json ... ```
    match_code_block = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if match_code_block:
        try:
            val = orjson.loads(match_code_block.group(1).strip())
            if isinstance(val, list) and len(val) > 0 and isinstance(val[0], dict):
                return val[0]
            if isinstance(val, dict):
                return val
        except Exception:
            pass

    # 2. Tìm kiếm cặp ngoặc nhọn JSON { ... }
    match_object = re.search(r"(\{[\s\S]*\})", text)
    if match_object:
        try:
            val = orjson.loads(match_object.group(1).strip())
            if isinstance(val, dict):
                return val
        except Exception:
            pass

    # 3. Phân tích Regex dự phòng nếu AI sinh phản hồi không đúng chuẩn JSON
    is_invalid = re.search(
        r"(?:is_valid_leader|status)[\"'\s:]+(?:false|unverified|no|rejected)",
        text,
        re.IGNORECASE
    )
    is_valid = re.search(
        r"(?:is_valid_leader|status)[\"'\s:]+(?:true|verified|yes)",
        text,
        re.IGNORECASE
    )
    if is_invalid and not is_valid:
        return {"is_valid_leader": False}

    name_match = re.search(r"(?:name|actual_ceo)[\"'\s:]+[\"']([^\"'\n]+)[\"']", text, re.IGNORECASE)
    role_match = re.search(r"role[\"'\s:]+[\"']([^\"'\n]+)[\"']", text, re.IGNORECASE)
    comp_match = re.search(r"company[\"'\s:]+[\"']([^\"'\n]+)[\"']", text, re.IGNORECASE)

    if name_match or is_valid:
        return {
            "is_valid_leader": True,
            "name": name_match.group(1).strip() if name_match else "",
            "role": role_match.group(1).strip() if role_match else "CEO",
            "company": comp_match.group(1).strip() if comp_match else "",
        }

    return None


def verify_executive_with_gemini(
    name: str,
    role: str,
    company: str,
    target: str,
    snippet: str = "",
    gemini_key: Optional[str] = None,
    gemini_model: Optional[str] = None,
    timeout: int = 15
) -> Tuple[bool, str, str, str, str]:
    """Sử dụng mô hình Google AI (Gemini / Gemma) để xác thực người này có phải lãnh đạo đương nhiệm hay không.

    Returns:
        (is_valid, verified_name, verified_role, verified_company, ai_status_text)
    """
    effective_key = (gemini_key or "").strip() or get_secret(
        ["GEMINI_API_KEY", "GEMINI_KEY", "GOOGLE_API_KEY"]
    )
    if not effective_key:
        return True, name, role, company, "Chưa xác minh (Regex)"

    model_name = (gemini_model or "gemma-4-26b-a4b-it").strip()
    api_url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={effective_key}"

    prompt = (
        f"Determine if {name} ({role} @ {company}) is currently a top executive/CEO/Founder "
        f"based on: {snippet[:350]}. Respond JSON: "
        f'{{{{ "is_valid_leader": true/false, "name": "{name}", "role": "{role}", "company": "{company}" }}}}'
    )

    try:
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0.0, "maxOutputTokens": 400},
        }
        response = requests.post(api_url, json=payload, timeout=timeout)
        if response.status_code == 200:
            candidates = response.json().get("candidates", [])
            if candidates:
                raw_text = (
                    candidates[0]
                    .get("content", {})
                    .get("parts", [{}])[0]
                    .get("text", "")
                )
                parsed = extract_json_from_ai_response(raw_text)
                if parsed:
                    final_name = str(parsed.get("name") or name).strip()
                    final_role = str(parsed.get("role") or role).strip()
                    final_comp = str(parsed.get("company") or company).strip()

                    if parsed.get("is_valid_leader", True):
                        print(f"  ✅ [AI Hợp lệ] {final_name} | {final_role} @ {final_comp}")
                        return True, final_name, final_role, final_comp, f"✅ Xác minh bởi {model_name}"
                    else:
                        print(f"  ❌ [AI Loại bỏ] {name} ({role} @ {company}) không phải lãnh đạo hiện tại.")
                        return False, name, role, company, f"❌ Bị loại bởi {model_name}"
    except Exception:
        pass

    return True, name, role, company, "⚠️ Chưa xác minh"


# ==============================================================================
# 6. 🔎 CÔNG CỤ TÌM KIẾM CEO (SEARCH ENGINES)
# ==============================================================================

def extract_ceo_from_website(
    domain: str,
    proxy: Optional[str] = None,
    seen_urls: Optional[Set[str]] = None,
    gemini_key: Optional[str] = None,
    gemini_model: Optional[str] = None
) -> Optional[CEOLead]:
    """Quét trang giới thiệu/ban lãnh đạo (/about, /leadership, /team) trên website công ty."""
    clean_domain = re.sub(r"^https?://(www\.)?", "", domain).strip("/").split("/")[0]
    if not clean_domain:
        return None

    for path in ["/about", "/leadership", "/team"]:
        url = f"https://{clean_domain}{path}"
        page_tree = http_get(url, proxy=proxy, timeout=3)
        if not page_tree:
            continue

        for anchor in page_tree.css('a[href*="linkedin.com/in/"]'):
            href = anchor.attributes.get("href", "")
            match_link = re.search(
                r"(https?://[a-z]{0,3}\.?linkedin\.com/in/[a-zA-Z0-9_\-%]+)",
                href,
                re.IGNORECASE
            )
            if not match_link:
                continue

            clean_url = match_link.group(1).split("?")[0].rstrip("/")
            parent_tag = anchor.parent
            context_text = parent_tag.text(separator=" ", strip=True) if parent_tag else ""

            for keyword in CEO_KEYWORDS:
                if keyword in context_text.lower():
                    lines = [line.strip() for line in context_text.splitlines() if line.strip()]
                    fallback_name = clean_url.split("/in/")[-1].replace("-", " ").title()
                    candidate_name = lines[0] if lines and len(lines[0]) <= 50 else fallback_name

                    if is_duplicate_ceo(clean_url, candidate_name, seen_urls):
                        continue

                    is_valid, v_name, v_role, v_comp, ai_status = verify_executive_with_gemini(
                        name=candidate_name,
                        role=keyword.upper(),
                        company=clean_domain,
                        target=clean_domain,
                        snippet=context_text,
                        gemini_key=gemini_key,
                        gemini_model=gemini_model,
                    )
                    if is_valid:
                        return CEOLead(
                            name=v_name,
                            role=v_role,
                            keyword=f"Domain: {clean_domain}",
                            company=v_comp,
                            linkedin_link=clean_url,
                            ai_status=ai_status,
                        )
    return None


def search_ceo_for_domain(
    domain: str,
    proxy: Optional[str] = None,
    seen_urls: Optional[Set[str]] = None,
    gemini_key: Optional[str] = None,
    gemini_model: Optional[str] = None
) -> Optional[CEOLead]:
    """Tìm kiếm CEO theo tên miền website bằng giải thuật đa tầng (Multi-Tier Executive Resolution):

    - Tầng 1: Executive Search - Tìm danh tính CEO thực tế từ tri thức toàn cầu / Wikipedia.
    - Tầng 2: Targeted LinkedIn Lookup - Truy vấn chính xác LinkedIn profile của vị CEO đó.
    - Tầng 3: Direct LinkedIn Query - Fallback trực tiếp cho các công ty vừa và nhỏ.
    """
    clean_domain = re.sub(r"^https?://(www\.)?", "", domain).strip("/").split("/")[0]
    raw_brand = clean_domain.split(".")[0].title()
    brand_name = BRAND_MAP.get(raw_brand, raw_brand)
    ceo_name = None

    # TẦNG 1: Executive Search - Xác thực danh tính CEO thực tế
    q1 = f'"current CEO of {brand_name}" OR "CEO of {brand_name}" OR "Chief Executive Officer of {brand_name}"'
    search_url_1 = f"https://search.yahoo.com/search?p={urllib.parse.quote(q1)}"
    tree_exec = http_get(search_url_1, proxy=proxy, timeout=10)

    if tree_exec:
        for li in tree_exec.css("div#web ol > li")[:4]:
            h3_tag = li.css_first("h3")
            desc_tag = li.css_first("div.compText") or li.css_first("p")
            title_text = h3_tag.text(strip=True) if h3_tag else ""
            snippet_text = desc_tag.text(strip=True) if desc_tag else ""

            # Nhận diện mẫu Wikipedia: 'Sundar Pichai - Wikipedia'
            wiki_match = re.search(r"^([A-ZÀ-Ỹ][a-zA-ZÀ-ỹ\.\s]{2,30})\s*[-–|]\s*Wikipedia", title_text)
            if wiki_match and not any(k in wiki_match.group(1).lower() for k in ["list", "history", brand_name.lower()]):
                ceo_name = wiki_match.group(1).strip()
                break

            # Nhận diện mẫu câu '[Tên] is President and CEO of [Brand]'
            ceo_match = re.search(
                r"([A-ZÀ-Ỹ][a-zA-ZÀ-ỹ\.\s]{2,30})\s+(?:is\s+(?:the\s+)?(?:President\s+and\s+)?(?:CEO|Chief Executive Officer)\s+of|là\s+Tổng\s+Giám\s+Đốc\s+của)\s+"
                + re.escape(brand_name),
                f"{title_text}. {snippet_text}",
                re.IGNORECASE
            )
            if ceo_match:
                candidate = ceo_match.group(1).strip()
                if len(candidate.split()) <= 4 and not any(w in candidate.lower() for w in ["who", "he", "she", "former"]):
                    ceo_name = candidate
                    break

    # TẦNG 2: Targeted LinkedIn Lookup - Truy xuất LinkedIn của vị CEO đã xác định
    if ceo_name:
        q_linkedin = f'site:linkedin.com/in/ "{ceo_name}" "{brand_name}"'
        search_url_li = f"https://search.yahoo.com/search?p={urllib.parse.quote(q_linkedin)}"
        tree_linkedin = http_get(search_url_li, proxy=proxy, timeout=10)

        if tree_linkedin:
            for li in tree_linkedin.css("div#web ol > li")[:3]:
                anchor = li.css_first('a[href*="RU="]') or li.css_first('a[href*="linkedin.com"]')
                if not anchor:
                    continue

                clean_url = unquote_yahoo_url(anchor.attributes.get("href", ""))
                if "linkedin.com/in/" in clean_url:
                    h3_tag = li.css_first("h3")
                    desc_tag = li.css_first("div.compText") or li.css_first("p")
                    title_text = h3_tag.text(strip=True) if h3_tag else ""
                    snippet_text = desc_tag.text(strip=True) if desc_tag else ""

                    is_valid, v_name, v_role, v_comp, ai_status = verify_executive_with_gemini(
                        name=ceo_name,
                        role="CEO",
                        company=brand_name,
                        target=brand_name,
                        snippet=f"{title_text}. {snippet_text}",
                        gemini_key=gemini_key,
                        gemini_model=gemini_model,
                    )
                    if is_valid:
                        return CEOLead(
                            name=v_name,
                            role=v_role,
                            keyword=f"Domain: {clean_domain}",
                            company=v_comp,
                            linkedin_link=clean_url,
                            ai_status=ai_status,
                        )

    # TẦNG 3: Direct LinkedIn Query - Fallback trực tiếp
    q_direct = (
        f'site:linkedin.com/in/ ("CEO" OR "Chief Executive Officer" OR "Founder") '
        f'("{clean_domain}" OR "{brand_name}") -intitle:former -intitle:ex'
    )
    search_url_direct = f"https://search.yahoo.com/search?p={urllib.parse.quote(q_direct)}"
    tree_direct = http_get(search_url_direct, proxy=proxy, timeout=12)

    if tree_direct:
        for li in tree_direct.css("div#web ol > li")[:5]:
            h3_tag = li.css_first("h3")
            anchor = li.css_first('a[href*="RU="]') or li.css_first('a[href*="linkedin.com"]')
            if not h3_tag or not anchor:
                continue

            clean_url = unquote_yahoo_url(anchor.attributes.get("href", ""))
            if "linkedin.com/in/" not in clean_url:
                continue

            desc_tag = li.css_first("div.compText") or li.css_first("p")
            title_text = h3_tag.text(strip=True)
            snippet_text = desc_tag.text(strip=True) if desc_tag else ""

            candidate_name, candidate_role, candidate_company = parse_linkedin_title(
                title_text, snippet=snippet_text
            )
            if not candidate_name or not is_leadership_role(candidate_role):
                continue

            final_company = candidate_company if candidate_company not in ("", "N/A") else brand_name
            is_valid, v_name, v_role, v_comp, ai_status = verify_executive_with_gemini(
                name=candidate_name,
                role=candidate_role,
                company=final_company,
                target=brand_name,
                snippet=snippet_text,
                gemini_key=gemini_key,
                gemini_model=gemini_model,
            )
            if is_valid:
                return CEOLead(
                    name=v_name,
                    role=v_role,
                    keyword=f"Domain: {clean_domain}",
                    company=v_comp,
                    linkedin_link=clean_url,
                    ai_status=ai_status,
                )

    return None


def search_ceos_by_keyword(
    keyword: str,
    target_count: int = 10,
    proxy: Optional[str] = None,
    seen_urls: Optional[Set[str]] = None,
    skip_empty_company: bool = False,
    max_pages: int = 10,
    gemini_key: Optional[str] = None,
    gemini_model: Optional[str] = None
) -> List[CEOLead]:
    """Tìm kiếm danh sách CEO / Lãnh đạo trên LinkedIn theo từ khóa ngành nghề."""
    seen_urls = seen_urls if seen_urls is not None else set()
    leads: List[CEOLead] = []
    page = 0

    while len(leads) < target_count and page < max_pages:
        offset = page * 7 + 1
        page += 1

        if " " in keyword:
            keyword_query = " ".join(f'"{word}"' for word in keyword.strip().split() if len(word) > 1)
        else:
            keyword_query = f'"{keyword}"'

        query_string = (
            f'site:linkedin.com/in/ ("CEO" OR "Chief Executive Officer" OR "Founder") '
            f'{keyword_query} -intitle:former -intitle:ex'
        )
        search_url = f"https://search.yahoo.com/search?p={urllib.parse.quote(query_string)}&b={offset}"
        page_tree = http_get(search_url, proxy=proxy, timeout=12)
        if not page_tree:
            time.sleep(0.4)
            continue

        results = page_tree.css("div#web ol > li")
        if not results:
            time.sleep(0.4)
            continue

        for li in results:
            h3_tag = li.css_first("h3")
            anchor = li.css_first('a[href*="RU="]') or li.css_first('a[href*="linkedin.com"]')
            if not h3_tag or not anchor:
                continue

            clean_url = unquote_yahoo_url(anchor.attributes.get("href", ""))
            if "linkedin.com/in/" not in clean_url or clean_url in seen_urls:
                continue

            desc_tag = li.css_first("div.compText") or li.css_first("p")
            title_text = h3_tag.text(strip=True)
            snippet_text = desc_tag.text(strip=True) if desc_tag else ""

            candidate_name, candidate_role, candidate_company = parse_linkedin_title(
                title_text, snippet=snippet_text
            )
            if not candidate_name or len(candidate_name) <= 2 or not is_leadership_role(candidate_role):
                continue
            if skip_empty_company and candidate_company in ("", "N/A"):
                continue
            if is_duplicate_ceo(clean_url, candidate_name, seen_urls):
                continue

            is_valid, v_name, v_role, v_comp, ai_status = verify_executive_with_gemini(
                name=candidate_name,
                role=candidate_role,
                company=candidate_company,
                target=keyword,
                snippet=snippet_text,
                gemini_key=gemini_key,
                gemini_model=gemini_model,
            )
            if not is_valid:
                continue

            # Đánh dấu đã thấy để chống trùng
            seen_urls.add(clean_url)
            match_username = re.search(r"linkedin\.com/in/([^/]+)", clean_url, re.IGNORECASE)
            if match_username:
                seen_urls.add(match_username.group(1).lower())
            seen_urls.add(v_name.strip().lower())

            lead = CEOLead(
                name=v_name,
                role=v_role,
                keyword=keyword,
                company=v_comp,
                linkedin_link=clean_url,
                ai_status=ai_status,
            )
            leads.append(lead)
            print_lead(lead, len(leads))

            if len(leads) >= target_count:
                break

        time.sleep(0.4)

    return leads


# ==============================================================================
# 7. 📊 TỰ ĐỘNG ĐỊNH DẠNG & LƯU GOOGLE SHEETS (SHEETS ENGINE)
# ==============================================================================

def format_sheet(ws: Any) -> None:
    """Tự động làm đẹp trang tính: Cố định dòng đầu, đặt màu nền, căn giữa STT/Ngày, và chỉnh độ rộng cột."""
    try:
        # Cố định dòng tiêu đề
        ws.freeze(rows=1)

        # Định dạng Header (Màu xanh đậm doanh nghiệp, chữ trắng in đậm)
        ws.format("A1:H1", {
            "backgroundColor": {"red": 0.12, "green": 0.31, "blue": 0.47},
            "horizontalAlignment": "CENTER",
            "verticalAlignment": "MIDDLE",
            "textFormat": {
                "foregroundColor": {"red": 1.0, "green": 1.0, "blue": 1.0},
                "bold": True,
                "fontSize": 10
            }
        })

        # Căn lề các dòng dữ liệu
        last_row = len(ws.col_values(1))
        if last_row > 1:
            ws.format(f"A2:A{last_row}", {"horizontalAlignment": "CENTER", "verticalAlignment": "MIDDLE"})
            ws.format(f"G2:H{last_row}", {"horizontalAlignment": "CENTER", "verticalAlignment": "MIDDLE"})
            ws.format(f"B2:F{last_row}", {"verticalAlignment": "MIDDLE"})

        # Điều chỉnh độ rộng từng cột
        requests_body = [
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": ws.id,
                        "dimension": "COLUMNS",
                        "startIndex": i,
                        "endIndex": i + 1,
                    },
                    "properties": {"pixelSize": width},
                    "fields": "pixelSize",
                }
            }
            for i, width in enumerate(SHEET_COL_WIDTHS)
        ]
        ws.spreadsheet.batch_update({"requests": requests_body})
    except Exception as exc:
        print(f"⚠️ Cảnh báo khi format Sheet: {exc}")


def renumber_sheet(ws: Any) -> None:
    """Đánh lại số thứ tự (STT) cột A từ 1 đến N liên tục."""
    try:
        all_rows = ws.get_all_values()
        if len(all_rows) <= 1:
            return
        data_rows = [row for row in all_rows[1:] if any(row)]
        for idx, row in enumerate(data_rows, start=1):
            row[0] = str(idx)
        ws.update(values=data_rows, range_name=f"A2:H{len(data_rows) + 1}")
    except Exception as exc:
        print(f"⚠️ Cảnh báo khi đánh lại STT Sheet: {exc}")


def get_existing_sheet_ceos(ws: Any) -> Set[str]:
    """Đọc dữ liệu hiện có trong Sheet để lập danh sách đen chống cào trùng lặp."""
    if not ws:
        return set()
    try:
        values = ws.get_all_values()
        if len(values) <= 1:
            return set()

        headers = [h.strip().lower() for h in values[0]]
        linkedin_col_idx = next((i for i, h in enumerate(headers) if "linkedin" in h), -1)
        name_col_idx = next((i for i, h in enumerate(headers) if any(k in h for k in ["name", "tên", "ceo"])), -1)

        seen: Set[str] = set()
        for row in values[1:]:
            if linkedin_col_idx != -1 and len(row) > linkedin_col_idx and row[linkedin_col_idx].strip():
                clean_u = row[linkedin_col_idx].strip().split("?")[0].rstrip("/").lower()
                seen.add(clean_u)
                match_user = re.search(r"linkedin\.com/in/([^/]+)", clean_u, re.IGNORECASE)
                if match_user:
                    seen.add(match_user.group(1).lower())
            if name_col_idx != -1 and len(row) > name_col_idx and row[name_col_idx].strip():
                seen.add(row[name_col_idx].strip().lower())
        return seen
    except Exception:
        return set()


def connect_google_sheet(sheet_id: str = "", worksheet_name: str = "sheet1") -> Optional[Any]:
    """Kết nối với Google Sheet bằng xác thực Colab hoặc Service Account và chuẩn hóa cấu trúc 8 cột."""
    sid = (sheet_id or "").strip() or get_secret(["SHEET_ID", "GOOGLE_SHEET_ID", "SPREADSHEET_ID"])
    if not sid:
        print("❌ LỖI: Bạn chưa điền SHEET_ID. Vui lòng dán ID Sheet vào ô cấu hình.")
        return None

    target_ws_name = (worksheet_name or "sheet1").strip()
    gc = None

    # 1. Thử xác thực tự động trong môi trường Google Colab
    if IN_COLAB:
        try:
            from google.auth import default
            credentials, _ = default(
                scopes=[
                    "https://www.googleapis.com/auth/spreadsheets",
                    "https://www.googleapis.com/auth/drive"
                ]
            )
            gc = gspread.authorize(credentials)
        except Exception:
            try:
                from google.colab import auth
                auth.authenticate_user()
                from google.auth import default
                credentials, _ = default()
                gc = gspread.authorize(credentials)
            except Exception:
                pass

    # 2. Thử xác thực bằng file service_account.json cục bộ
    if not gc:
        for sa_path in ["service_account.json", "../service_account.json"]:
            if os.path.exists(sa_path):
                try:
                    gc = gspread.service_account(filename=sa_path)
                    break
                except Exception:
                    pass

    if not gc:
        print("❌ Không tìm thấy thông tin đăng nhập Google. Hãy chạy BƯỚC 2 để đăng nhập!")
        return None

    # Mở Spreadsheet
    try:
        sh = gc.open_by_key(sid)
        print(f"✅ Đã kết nối Spreadsheet: '{sh.title}'")
    except Exception as exc:
        print(f"❌ Lỗi mở Spreadsheet ID '{sid}': {exc}")
        return None

    # Mở hoặc tạo mới Worksheet
    try:
        try:
            ws = sh.worksheet(target_ws_name)
        except gspread.WorksheetNotFound:
            ws = sh.add_worksheet(title=target_ws_name, rows=100, cols=10)
            print(f"✅ Đã tạo tab mới: '{target_ws_name}'")

        # Kiểm tra và nâng cấp Headers lên chuẩn 8 cột
        all_vals = ws.get_all_values()
        if not all_vals:
            ws.update(values=[SHEET_HEADERS], range_name="A1")
            print("✅ Đã khởi tạo cấu trúc 8 cột chuẩn.")
        elif all_vals[0] != SHEET_HEADERS:
            migrated_rows = []
            for i, r in enumerate(all_vals[1:], start=1):
                if not any(r):
                    continue
                ai_val = r[6] if len(r) >= 8 else "⚠️ Chưa xác minh"
                date_val = r[7] if len(r) >= 8 else (r[6] if len(r) == 7 else now_hcm_str())
                migrated_rows.append([
                    str(i),
                    r[1] if len(r) > 1 else "",
                    r[2] if len(r) > 2 else "",
                    r[3] if len(r) > 3 else "",
                    r[4] if len(r) > 4 else "",
                    r[5] if len(r) > 5 else "",
                    ai_val,
                    date_val,
                ])
            ws.clear()
            ws.update(values=[SHEET_HEADERS] + migrated_rows, range_name=f"A1:H{len(migrated_rows) + 1}")

        renumber_sheet(ws)
        format_sheet(ws)
        return ws
    except Exception as exc:
        print(f"❌ Lỗi thao tác Worksheet '{target_ws_name}': {exc}")
        return None


def save_leads_to_sheet(
    leads: List[Union[CEOLead, Dict[str, Any]]],
    worksheet: Optional[Any] = None,
    sheet_id: str = "",
    worksheet_name: str = "sheet1"
) -> bool:
    """Ghi danh sách CEO mới lên ĐẦU trang tính (sau header) và tự động căn chỉnh STT."""
    if not leads:
        return False

    ws = worksheet or connect_google_sheet(sheet_id=sheet_id, worksheet_name=worksheet_name)
    if not ws:
        return False

    try:
        all_rows = ws.get_all_values()

        # Dữ liệu mới cần lưu
        new_data = []
        for item in leads:
            lead = item.to_dict() if isinstance(item, CEOLead) else item
            new_data.append([
                lead.get("name", ""),
                lead.get("role", ""),
                lead.get("keyword", ""),
                lead.get("company", ""),
                lead.get("linkedin_link") or lead.get("url") or "",
                lead.get("ai_status") or "⚠️ Chưa xác minh",
                lead.get("updated_at") or now_hcm_str(),
            ])

        # Dữ liệu cũ đã có trong Sheet (bỏ cột STT để đánh lại đồng bộ)
        existing_data = []
        for r in all_rows[1:]:
            if any(r):
                if len(r) >= 8:
                    row_info = list(r[1:8])
                elif len(r) == 7:
                    row_info = [r[1], r[2], r[3], r[4], r[5], "⚠️ Chưa xác minh", r[6]]
                else:
                    row_info = list(r[1:])
                    while len(row_info) < 7:
                        row_info.append("")
                existing_data.append(row_info)

        # Ghép dữ liệu mới lên trên cùng
        combined = new_data + existing_data
        final_rows = [[i] + row for i, row in enumerate(combined, start=1)]

        ws.update(values=final_rows, range_name=f"A2:H{len(final_rows) + 1}")
        format_sheet(ws)
        print(f"\n💾 ĐÃ LƯU THÀNH CÔNG {len(leads)} CEO MỚI LÊN ĐẦU SHEET (Tổng: {len(final_rows)} dòng).")
        return True
    except Exception as exc:
        print(f"❌ Lỗi ghi dữ liệu vào Sheet: {exc}")
        return False


# ==============================================================================
# 8. 🚀 ĐIỀU PHỐI TÌM KIẾM CẤP CAO (ORCHESTRATORS & RUNNER)
# ==============================================================================

def find_ceos_by_domains(
    company_domains: Union[List[str], str],
    target_count: Optional[int] = None,
    save_to_sheet: bool = True,
    max_workers: int = 5,
    sheet_id: str = "",
    worksheet_name: str = "sheet1",
    proxy: Optional[str] = None,
    gemini_key: Optional[str] = None,
    gemini_model: Optional[str] = None
) -> List[CEOLead]:
    """Tìm kiếm danh tính CEO theo danh sách tên miền website đa luồng."""
    clean_sid = (sheet_id or "").strip() or get_secret(["SHEET_ID", "GOOGLE_SHEET_ID", "SPREADSHEET_ID"])
    clean_gemini_key = (gemini_key or "").strip() or get_secret(["GEMINI_API_KEY", "GEMINI_KEY", "GOOGLE_API_KEY"]) or None
    effective_proxy = (proxy or "").strip() or get_secret(["HTTP_PROXY", "HTTPS_PROXY", "PROXY"]) or None

    # Tách danh sách tên miền từ chuỗi hoặc danh sách
    if isinstance(company_domains, str):
        raw_list = [d.strip() for d in re.split(r"[,\n\r;]+", company_domains) if d.strip()]
    else:
        raw_list = list(company_domains)

    clean_domains: List[str] = []
    seen_domains: Set[str] = set()
    for d in raw_list:
        clean_d = re.sub(r"^https?://(www\.)?", "", d).strip("/").split("/")[0]
        if clean_d and clean_d.lower() not in seen_domains:
            seen_domains.add(clean_d.lower())
            clean_domains.append(clean_d)

    if not clean_domains:
        print("⚠️ Không có tên miền hợp lệ để quét.")
        return []

    print(f"\n🌐 BẮT ĐẦU TÌM CEO CHO {len(clean_domains)} WEBSITE [Số luồng: {max_workers}]", flush=True)
    leads: List[CEOLead] = []
    seen_urls: Set[str] = set()
    ws = None

    if save_to_sheet and clean_sid:
        ws = connect_google_sheet(sheet_id=clean_sid, worksheet_name=worksheet_name)
        if ws:
            seen_urls.update(get_existing_sheet_ceos(ws))

    def _fetch_domain_worker(index: int, domain_name: str):
        time.sleep(index * 0.15)
        # 1. Thử thuật toán tìm kiếm đa tầng Yahoo Search
        result = search_ceo_for_domain(
            domain_name,
            proxy=effective_proxy,
            seen_urls=seen_urls,
            gemini_key=clean_gemini_key,
            gemini_model=gemini_model
        )
        # 2. Nếu chưa thấy, thử cào trực tiếp trang /about, /team của website
        if not result:
            result = extract_ceo_from_website(
                domain_name,
                proxy=effective_proxy,
                seen_urls=seen_urls,
                gemini_key=clean_gemini_key,
                gemini_model=gemini_model
            )
        return domain_name, result

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_domain = {
            executor.submit(_fetch_domain_worker, i, dom): dom
            for i, dom in enumerate(clean_domains)
        }
        for future in as_completed(future_to_domain):
            domain_item = future_to_domain[future]
            try:
                _, result_lead = future.result()
                if not result_lead:
                    print(f"  ⚠️ Chưa tìm thấy CEO phù hợp: {domain_item}")
                    continue

                if is_duplicate_ceo(result_lead.linkedin_link, result_lead.name, seen_urls):
                    print(f"  ⏭️ Đã có trong Sheet: {result_lead.name} ({domain_item})")
                    continue

                seen_urls.add(result_lead.linkedin_link)
                match_u = re.search(r"linkedin\.com/in/([^/]+)", result_lead.linkedin_link, re.IGNORECASE)
                if match_u:
                    seen_urls.add(match_u.group(1).lower())
                seen_urls.add(result_lead.name.strip().lower())

                leads.append(result_lead)
                print_lead(result_lead, len(leads), prefix="Domain")

                if target_count and len(leads) >= target_count:
                    break
            except Exception as exc:
                print(f"  ❌ Lỗi khi quét {domain_item}: {exc}")

    print(f"\n🎯 Tổng cộng tìm thấy: {len(leads)} CEO từ tên miền.")
    if save_to_sheet and leads and clean_sid:
        save_leads_to_sheet(leads, worksheet=ws, sheet_id=clean_sid, worksheet_name=worksheet_name)

    return leads


def find_ceos_by_keyword(
    keyword: Optional[str] = None,
    keywords: Optional[Union[List[str], str]] = None,
    target_count: int = 10,
    save_to_sheet: bool = True,
    max_workers: int = 5,
    sheet_id: str = "",
    worksheet_name: str = "sheet1",
    proxy: Optional[str] = None,
    skip_empty_company: bool = False,
    max_pages: int = 10,
    gemini_key: Optional[str] = None,
    gemini_model: Optional[str] = None
) -> List[CEOLead]:
    """Tìm kiếm CEO theo từ khóa ngành nghề đa luồng."""
    raw = keywords if keywords is not None else keyword
    if isinstance(raw, str):
        candidates = [k.strip() for k in re.split(r"[,\n\r;]+", raw) if k.strip()]
    elif isinstance(raw, (list, set, tuple)):
        candidates = [str(k).strip() for k in raw if str(k).strip()]
    else:
        candidates = []

    if keyword and isinstance(keyword, str) and keyword.strip() and keyword.strip() not in candidates:
        candidates.append(keyword.strip())

    clean_keywords: List[str] = []
    seen_kw: Set[str] = set()
    for kw in candidates:
        if kw and kw.lower() not in seen_kw:
            seen_kw.add(kw.lower())
            clean_keywords.append(kw)

    if not clean_keywords:
        clean_keywords = ["Fintech Vietnam"]

    print(f"\n🚀 BẮT ĐẦU TÌM KIẾM CEO [Từ khóa: '{', '.join(clean_keywords)}' | Mục tiêu: {target_count}/kw]", flush=True)
    leads: List[CEOLead] = []
    seen_urls: Set[str] = set()
    ws = None

    clean_sid = (sheet_id or "").strip() or get_secret(["SHEET_ID", "GOOGLE_SHEET_ID", "SPREADSHEET_ID"])
    if save_to_sheet and clean_sid:
        ws = connect_google_sheet(sheet_id=clean_sid, worksheet_name=worksheet_name)
        if ws:
            seen_urls.update(get_existing_sheet_ceos(ws))

    # Nếu chỉ có 1 từ khóa, chạy tuần tự để tối ưu tốc độ và output
    if len(clean_keywords) == 1:
        leads = search_ceos_by_keyword(
            keyword=clean_keywords[0],
            target_count=target_count,
            proxy=proxy,
            seen_urls=seen_urls,
            skip_empty_company=skip_empty_company,
            max_pages=max_pages,
            gemini_key=gemini_key,
            gemini_model=gemini_model,
        )
    else:
        def _fetch_kw_worker(idx: int, kw_item: str) -> List[CEOLead]:
            time.sleep(idx * 0.2)
            return search_ceos_by_keyword(
                keyword=kw_item,
                target_count=target_count,
                proxy=proxy,
                seen_urls=seen_urls,
                skip_empty_company=skip_empty_company,
                max_pages=max_pages,
                gemini_key=gemini_key,
                gemini_model=gemini_model,
            )

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_kw = {
                executor.submit(_fetch_kw_worker, i, kw): kw
                for i, kw in enumerate(clean_keywords)
            }
            for future in as_completed(future_to_kw):
                try:
                    for lead in future.result():
                        if not is_duplicate_ceo(lead.linkedin_link, lead.name, seen_urls):
                            seen_urls.add(lead.linkedin_link)
                            seen_urls.add(lead.name.strip().lower())
                            leads.append(lead)
                except Exception as exc:
                    print(f"  ❌ Lỗi khi tìm kiếm từ khóa '{future_to_kw[future]}': {exc}")

    print(f"\n🎯 Tổng cộng tìm thấy: {len(leads)} CEO mới từ từ khóa.")
    if save_to_sheet and leads and clean_sid:
        save_leads_to_sheet(leads, worksheet=ws, sheet_id=clean_sid, worksheet_name=worksheet_name)

    return leads


def run_ceo_finder(
    sheet_id: str = "",
    worksheet_name: str = "sheet1",
    mode: str = "Keywords",
    keyword: str = "",
    websites: str = "",
    target_count: int = 5,
    skip_empty_company: bool = False,
    gemini_key: str = "",
    gemini_model: str = "gemma-4-26b-a4b-it",
    max_workers: int = 5,
    proxy: str = ""
) -> List[CEOLead]:
    """Hàm điều khiển cấp cao: Tự nhận diện Secrets, kết nối Sheet và khởi chạy tìm kiếm."""
    print("=" * 70)
    print("🚀 KHỞI ĐỘNG TIẾN TRÌNH TÌM KIẾM CEO")
    print("=" * 70)

    # 1. Tự động nhận diện Secrets nếu người dùng để trống
    clean_sid = (sheet_id or "").strip() or get_secret(["SHEET_ID", "GOOGLE_SHEET_ID", "SPREADSHEET_ID"])
    if clean_sid and not (sheet_id or "").strip():
        print("🔑 [Colab Secrets] Đã tự động nhận diện SHEET_ID.")

    clean_gemini_key = (gemini_key or "").strip() or get_secret(["GEMINI_API_KEY", "GEMINI_KEY", "GOOGLE_API_KEY"]) or None
    if clean_gemini_key and not (gemini_key or "").strip():
        print("🔑 [Colab Secrets] Đã tự động nhận diện GEMINI_API_KEY (kích hoạt Xác Minh AI).")
    elif not clean_gemini_key:
        print("ℹ️ Không dùng AI -> Chạy chế độ lọc Regex tiêu chuẩn.")

    effective_proxy = (proxy or "").strip() or get_secret(["HTTP_PROXY", "HTTPS_PROXY", "PROXY"]) or None
    if effective_proxy and not (proxy or "").strip():
        print("🔑 [Colab Secrets] Đã tự động nhận diện HTTP_PROXY.")

    # 2. Kiểm tra tính hợp lệ của SHEET_ID
    if not clean_sid:
        print("\n❌ BẠN CHƯA CÓ SHEET_ID!")
        print("👉 Cách 1: Dán ID vào ô 'SHEET_ID' ở bảng điều khiển bên dưới.")
        print("👉 Cách 2: Thêm Secret vào Colab (biểu tượng chìa khóa 🔑 ở thanh bên trái):")
        print("          - Name: SHEET_ID")
        print("          - Value: <ID Google Sheet của bạn>")
        print("          - Bật công tắc 'Notebook access' (Quyền truy cập của sổ tay)")
        return []

    # 3. Kiểm tra kết nối Google Sheet
    print(f"\n⏳ Đang kiểm tra kết nối Google Sheet (ID: {clean_sid[:10]}...)...\n")
    ws = connect_google_sheet(sheet_id=clean_sid, worksheet_name=worksheet_name)
    if not ws:
        print("\n⚠️ Không thể tiếp tục do chưa kết nối được Google Sheet. Hãy kiểm tra lại quyền truy cập hoặc ID Sheet!")
        return []

    # 4. Thực thi tìm kiếm theo chế độ người dùng chọn
    results: List[CEOLead] = []
    if mode == "Keywords":
        results = find_ceos_by_keyword(
            keywords=keyword,
            target_count=int(target_count),
            save_to_sheet=True,
            max_workers=int(max_workers),
            sheet_id=clean_sid,
            worksheet_name=worksheet_name,
            proxy=effective_proxy,
            skip_empty_company=skip_empty_company,
            gemini_key=clean_gemini_key,
            gemini_model=gemini_model,
        )
    else:
        results = find_ceos_by_domains(
            company_domains=websites,
            save_to_sheet=True,
            max_workers=int(max_workers),
            sheet_id=clean_sid,
            worksheet_name=worksheet_name,
            proxy=effective_proxy,
            gemini_key=clean_gemini_key,
            gemini_model=gemini_model,
        )

    print("\n" + "=" * 70)
    print(f"🎉 HOÀN THÀNH XUẤT SẮC! Đã tìm và lưu {len(results)} CEO mới.")
    print(f"📊 Mời bạn mở link Google Sheet để xem kết quả:")
    print(f"👉 https://docs.google.com/spreadsheets/d/{clean_sid}/edit")
    print("=" * 70)
    return results


print("✅ BỘ XỬ LÝ TRUNG TÂM ĐÃ SẴN SÀNG!")


In [ ]:
#@title 🎯 BẢNG ĐIỀU KHIỂN TÌM KIẾM CEO { display-mode: "form" }
#@markdown > 💡 **Mẹo lưu bảo mật với Colab Secrets (🔑):**
#@markdown > Bạn có thể lưu trước các thông tin bảo mật vào mục **Secrets** (biểu tượng chìa khóa 🔑 ở thanh bên trái Colab) và bật **Notebook access**:
#@markdown > - `SHEET_ID`: ID của Google Sheet cần lưu kết quả.
#@markdown > - `GEMINI_API_KEY`: API Key của Google Gemini.
#@markdown > - `HTTP_PROXY`: Proxy nếu cần (tùy chọn).
#@markdown >
#@markdown > *Nếu đã lưu trong Colab Secrets, bạn có thể **để trống** các ô tương ứng trong form dưới — hệ thống sẽ tự động nhận diện!*
#@markdown

#@markdown ---
#@markdown ### 📊 1. Cấu hình Google Sheet lưu kết quả
#@markdown *Dán ID của Google Sheet (chuỗi ký tự nằm giữa `/d/` và `/edit` trên URL trình duyệt).*
SHEET_ID = "" #@param {type:"string"}
WORKSHEET_NAME = "sheet1" #@param {type:"string"}
#@markdown

#@markdown ---
#@markdown ### 🔍 2. Chọn chế độ tìm kiếm và thông số
#@markdown - **Keywords**: Tìm kiếm lãnh đạo theo từ khóa hoặc ngành nghề (dùng ô `TU_KHOA` bên dưới).
#@markdown - **Domains**: Tìm kiếm lãnh đạo theo danh sách website công ty (dùng ô `DANH_SACH_WEBSITE` bên dưới).
CHE_DO_TIM_KIEM = "Keywords" #@param ["Keywords", "Domains"]
#@markdown

#@markdown ---
#@markdown ### 🎯 3. Thông số tìm kiếm
#@markdown *Điền từ khóa nếu chọn chế độ **Keywords**; điền danh sách website nếu chọn chế độ **Domains**.*
TU_KHOA = "Fintech" #@param {type:"string"}
DANH_SACH_WEBSITE = "https://www.nvidia.com, https://www.amd.com" #@param {type:"string"}
SO_LUONG_MOI_TU_KHOA = 5 #@param {type:"integer"}
BO_QUA_CONG_TY_TRONG = True #@param {type:"boolean"}
#@markdown

#@markdown ---
#@markdown ### 🧠 4. Trợ lý AI xác minh chức danh (Tùy chọn)
#@markdown *Nếu đã lưu `GEMINI_API_KEY` trong Secrets 🔑 thì có thể để trống. AI giúp lọc chuẩn CEO đương nhiệm.*
GEMINI_API_KEY = "" #@param {type:"string"}
GEMINI_MODEL = "gemma-4-26b-a4b-it" #@param ["gemma-4-26b-a4b-it", "gemma-4-31b-it"]
#@markdown

#@markdown ---
#@markdown ### ⚡ 5. Tốc độ & Kết nối mạng nâng cao
SO_LUONG_LUONG_SONG_SONG = 5 #@param {type:"slider", min:1, max:10, step:1}
HTTP_PROXY = "" #@param {type:"string"}

# ==============================================================================
# KHỞI CHẠY TÌM KIẾM
# ==============================================================================
run_ceo_finder(
    sheet_id=SHEET_ID,
    worksheet_name=WORKSHEET_NAME,
    mode=CHE_DO_TIM_KIEM,
    keyword=TU_KHOA,
    websites=DANH_SACH_WEBSITE,
    target_count=SO_LUONG_MOI_TU_KHOA,
    skip_empty_company=BO_QUA_CONG_TY_TRONG,
    gemini_key=GEMINI_API_KEY,
    gemini_model=GEMINI_MODEL,
    max_workers=SO_LUONG_LUONG_SONG_SONG,
    proxy=HTTP_PROXY
)
